In [ ]:
import numpy as np
from sympy import *

In [ ]:
# for two dependent variable distance D and price/unit P
# now optimized varibale w1,w2,b


# implemention usign sympy 

class MultipleRegression:

    def __init__(self,data,param):
        self.data = data
        self.param = param

    def cost(self):
        c = 0
        for i in range(len(self.data)):
            gather = 0
            for j in range(len(self.param)-1):
                gather += self.param[j]*self.data[i][j]

            c += (gather + self.param[-1] - self.data[i][-1])**2

        return c 


    def gradient(self):
        grad = []
        cost_fn = self.cost()

        for i in self.param:
            dw = diff(cost_fn,i)
            grad.append(dw)

        for g in range(len(grad)):
            grad[g] = lambdify([i for i in self.param],grad[g],'numpy')

        return grad 

    def grad_val(self,grad,param_val):
        values = []

        for g in grad:
            values.append(g(*param_val))

        return np.array(values, dtype=float) 


    def hessian(self):
        grad = []
        cost_fn = self.cost()

        for i in self.param:
            dw = diff(cost_fn,i)
            grad.append(dw)
            # print(dw)

        hess = []
        for g in grad:
            w = []
            for i in self.param:
                d = diff(g,i)
                d = lambdify([i for i in self.param],d,'numpy')
                w.append(d)

            hess.append(w)

        return hess 


    def hess_val(self,hess,param_val):
        val = []

        for h in hess:
            vali = []
            for hv in h:
                vali.append(hv(*param_val))

            val.append(vali)

        return val

    def gradient_descent(self,opt,alpha=0.1,iteration=100):
        g = self.gradient()
        for i in range(iteration):
            grad = self.grad_val(g,opt)
            
            for j in range(len(opt)):
                opt[j] = opt[j] - alpha*grad[j]
            print(opt)
        return opt

    

    def newton_method(self,opt,iteration):
        grad = self.gradient()
        hess = self.hessian()

        for i in range(iteration):
            grad_val = self.grad_val(grad,opt)
            hess_val = self.hess_val(hess,opt)

            opt -= np.linalg.inv(hess_val) @ grad_val
            print(opt)

        return opt 

In [ ]:
w1,w2,b = symbols('w1 w2 b')
param = [w1,w2,b]

def cost(x1,x2):
    return 2*x1 + 3*x2 + 4

data = [[1,2,0],[1,3,0],[3,7,0],[6,4,0],[4,9,0],[10,2,0],[12,15,0]]

for i in range(len(data)):
    data[i][2] = cost(data[i][0],data[i][1])

m = MultipleRegression(data,param)

opt = np.array([0,0,0],dtype=float)

In [ ]:
m.gradient_descent(opt,0.001,10000)

In [ ]:
# hess = m.hessian()
# arr = m.hess_val(opt)

# for row in arr:
#     print(*(f"{x:.2f}" for x in row))
opt = np.array([0,0,0],dtype=float)
m.newton_method(opt,5)

In [ ]:
# more general form for the Multiregression when the wn parameter need to oprimize for the n features (dependent variable)

class MultiRegressionGeneral:

    def __init__(self,data,output,param):
        self.data = np.asarray(data, dtype=float)
        self.output = np.asarray(output, dtype=float)
        self.param = np.asarray(param, dtype=float)

    def cost(self):
        prediction = (
            self.param[:-1, None] * self.data
        ).sum(axis=0) + self.param[-1]

        error = prediction - self.output

        return np.sum(error**2)

    def gradient(self):
        prediction = (
            self.param[:-1, None] * self.data
        ).sum(axis=0) + self.param[-1]

        error = prediction - self.output
        grad_w = 2 * (self.data @ error)
        grad_b = 2 * np.sum(error)

        return np.append(grad_w,grad_b)


    def gradient_descent(self,opt,alpha=0.1,iteration=100):
        self.param = np.asarray(opt,dtype=float)

        for i in range(iteration):
            grad = self.gradient()
            self.param -= alpha * grad
            print(self.param)

        return self.param
        

In [ ]:

def cost(x1,x2):
    return 2*x1 + 3*x2 + 4

distance = np.array([1,3,5,10,8,11,16,21,34,2,0],dtype=float)
price = np.array([12,3,6,8,9,0,5,6,14,1,56],dtype=float)

output = []
for i in range(len(distance)):
    output.append(cost(distance[i],price[i]))

data = [distance,price]
param = np.array([2,3,4],dtype=float)
m = MultiRegressionGeneral(data,output,param)

opt = np.array([0,0,0],dtype=float)

m.gradient_descent(opt,0.00001,200000)

In [ ]:

distance = np.array(
    [2, 5, 8, 3, 10, 15, 7, 12, 20, 6],
    dtype=float
)

price = np.array(
    [10, 15, 12, 20, 18, 25, 14, 30, 35, 16],
    dtype=float
)

area = np.array(
    [100, 150, 200, 120, 180, 250, 160, 220, 300, 140],
    dtype=float
)

age = np.array(
    [5, 3, 8, 2, 10, 4, 7, 1, 6, 9],
    dtype=float
)

rooms = np.array(
    [2, 3, 4, 2, 3, 5, 3, 4, 6, 2],
    dtype=float
)

data = [
    distance,
    price,
    area,
    age,
    rooms
]


output = (
    2 * distance
    + 3 * price
    + 0.5 * area
    - 4 * age
    + 10 * rooms
    + 20
)


param = np.zeros(6)

m2 = MultiRegressionGeneral(data,output,param)

m2.gradient_descent(param,0.000001,1000000)

# [2, 3, 0.5, -4, 10, 20]

In [ ]:

print(data)
print(output)
print(param)


print(m2.cost())

In [ ]:
print("Current:")
print(m2.param)
print("Cost:", m2.cost())
print("Gradient:", m2.gradient())

true_param2 = np.array([2., 3., 0.5, -4., 10., 20.])

m2.param = true_param2

print("\nTrue parameters:")
print(m2.param)
print("Cost:", m2.cost())
print("Gradient:", m2.gradient())

In [ ]:
# Now i reach a point where the scaling is required because when i increase the feature and they are differ by a vast range that lead to situation where if i decrease the learning rate than the increse overflow happen mean the it bound to another side of the palce and astart to reach infinity and if i decrease the rate that too small that make the roblem not reach to the optimized vaules due there differnce the scalling .

In [ ]:
# As in this problem our function is the quadratic applying the newton method gives the result constant so it reach to target in one iteration no matter hwo many parameter are you include . 